In [1]:
# create_results_table.py
import json
from pathlib import Path
import pandas as pd

def create_markdown_tables(results_dir=".", output_file="results_summary.md"):
    """
    Create markdown tables from YOLOv8 evaluation JSON files.
    
    Args:
        results_dir: Directory containing result folders (e.g., yolovX_widerperson_results)
        output_file: Output markdown file name
    """
    
    # Find all JSON files
    json_files = []
    for folder in Path(results_dir).glob("*_results"):
        json_files.extend(folder.glob("*.json"))
    
    if not json_files:
        print("No JSON files found!")
        return
    
    # Collect data
    data = []
    for json_file in sorted(json_files):
        with open(json_file, 'r') as f:
            result = json.load(f)
        
        # Extract model name from path or filename
        model_name = json_file.parent.name.replace("_widerperson_results", "")
        
        row = {
            'Model': model_name,
            'mAP@50': result.get('mAP50', result.get('mAP', {}).get('mAP50', 'N/A')),
            'mAP@50-95': result.get('mAP50-95', result.get('mAP', {}).get('mAP50-95', 'N/A')),
            'Precision': result.get('precision', result.get('mAP', {}).get('precision', 'N/A')),
            'Recall': result.get('recall', result.get('mAP', {}).get('recall', 'N/A')),
            'FPS': result.get('inference_speed', {}).get('fps', result.get('speed', {}).get('fps', 'N/A')),
            'Inference (ms)': result.get('inference_speed', {}).get('mean_time_ms', result.get('speed', {}).get('mean_ms', 'N/A')),
            'Total Detections': result.get('predictions', {}).get('total_detections', 'N/A'),
            'Avg Det/Image': result.get('predictions', {}).get('avg_detections_per_image', result.get('predictions', {}).get('avg_per_image', 'N/A'))
        }
        
        data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Sort by mAP@50 if available
    try:
        df['mAP@50_float'] = pd.to_numeric(df['mAP@50'], errors='coerce')
        df = df.sort_values('mAP@50_float', ascending=False)
        df = df.drop('mAP@50_float', axis=1)
    except:
        pass
    
    # Format numeric columns
    numeric_cols = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'FPS', 'Inference (ms)', 'Avg Det/Image']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f"{float(x):.4f}" if isinstance(x, (int, float)) and x != 'N/A' else x)
    
    # Create markdown content
    markdown = "# YOLOv8 WiderPerson Evaluation Results\n\n"
    
    # Main results table
    markdown += "## Detection Performance\n\n"
    markdown += df.to_markdown(index=False, floatfmt=".4f")
    
    # Create detailed speed comparison
    markdown += "\n\n## Inference Speed Details\n\n"
    
    speed_data = []
    for json_file in sorted(json_files):
        with open(json_file, 'r') as f:
            result = json.load(f)
        
        model_name = json_file.parent.name.replace("_widerperson_results", "")
        speed = result.get('inference_speed', result.get('speed', {}))
        
        if speed:
            speed_row = {
                'Model': model_name,
                'FPS': speed.get('fps', 'N/A'),
                'Mean (ms)': speed.get('mean_time_ms', speed.get('mean_ms', 'N/A')),
                'Std (ms)': speed.get('std_time_ms', speed.get('std_ms', 'N/A')),
                'Min (ms)': speed.get('min_time_ms', 'N/A'),
                'Max (ms)': speed.get('max_time_ms', 'N/A'),
                '95th %ile (ms)': speed.get('percentiles', {}).get('95th', 'N/A')
            }
            speed_data.append(speed_row)
    
    if speed_data:
        speed_df = pd.DataFrame(speed_data)
        
        # Format numeric columns
        for col in speed_df.columns:
            if col != 'Model':
                speed_df[col] = speed_df[col].apply(lambda x: f"{float(x):.2f}" if isinstance(x, (int, float)) and x != 'N/A' else x)
        
        # Sort by FPS
        try:
            speed_df['FPS_float'] = pd.to_numeric(speed_df['FPS'], errors='coerce')
            speed_df = speed_df.sort_values('FPS_float', ascending=False)
            speed_df = speed_df.drop('FPS_float', axis=1)
        except:
            pass
        
        markdown += speed_df.to_markdown(index=False, floatfmt=".2f")
    
    # Add model details if available
    markdown += "\n\n## Model Details\n\n"
    
    model_details = []
    for json_file in sorted(json_files):
        with open(json_file, 'r') as f:
            result = json.load(f)
        
        model_name = json_file.parent.name.replace("_widerperson_results", "")
        
        detail_row = {
            'Model': model_name,
            'Type': 'TensorRT' if result.get('is_tensorrt', False) else 'PyTorch',
            'Device': result.get('device', 'N/A'),
            'Batch Size': result.get('speed', {}).get('batch_size', result.get('inference_speed', {}).get('batch_size', 1))
        }
        
        # Add model path if available
        if 'model' in result:
            detail_row['Model Path'] = Path(result['model']).name
        
        model_details.append(detail_row)
    
    if model_details:
        details_df = pd.DataFrame(model_details)
        markdown += details_df.to_markdown(index=False)
    
    # Save to file
    with open(output_file, 'w') as f:
        f.write(markdown)
    
    print(f"Results saved to {output_file}")
    
    # Also create a CSV for easy import to Excel
    csv_file = output_file.replace('.md', '.csv')
    df.to_csv(csv_file, index=False)
    print(f"CSV saved to {csv_file}")

if __name__ == "__main__":
    # Assuming your result folders are in the current directory
    create_markdown_tables("EvalResults", "widerperson_results_summary.md")

Results saved to widerperson_results_summary.md
CSV saved to widerperson_results_summary.csv


In [2]:
# create_results_table.py
import json
from pathlib import Path
import pandas as pd
import re

def extract_model_info(model_name):
    """Extract YOLO version and size from model name."""
    # Pattern to match YOLOv8n, YOLOv11l, etc.
    pattern = r'(yolov?\d+)([nslmx])'
    match = re.search(pattern, model_name.lower())
    
    if match:
        version = match.group(1)
        size = match.group(2)
        return version, size
    return model_name, ''

def create_markdown_tables(results_dir=".", output_file="results_summary.md"):
    """
    Create markdown tables from YOLOv8 evaluation JSON files.
    Groups by YOLO version and excludes TensorRT models.
    """
    
    # Find all JSON files
    json_files = []
    for folder in Path(results_dir).glob("*_results"):
        # Skip folders with 'tensorrt' in name
        if 'tensorrt' in folder.name.lower():
            continue
        json_files.extend(folder.glob("*.json"))
    
    if not json_files:
        print("No JSON files found!")
        return
    
    # Collect data
    all_data = []
    for json_file in sorted(json_files):
        with open(json_file, 'r') as f:
            result = json.load(f)
        
        # Skip if model path contains tensorrt
        if 'model' in result and 'tensorrt' in str(result['model']).lower():
            continue
            
        # Skip if is_tensorrt flag is True
        if result.get('is_tensorrt', False):
            continue
        
        # Extract model name
        model_name = json_file.parent.name.replace("_widerperson_results", "")
        
        # Skip if model name contains tensorrt
        if 'tensorrt' in model_name.lower():
            continue
        
        row = {
            'Model': model_name,
            'mAP@50': result.get('mAP50', result.get('mAP', {}).get('mAP50', 'N/A')),
            'mAP@50-95': result.get('mAP50-95', result.get('mAP', {}).get('mAP50-95', 'N/A')),
            'mAP@75': result.get('mAP75', result.get('mAP', {}).get('mAP75', 'N/A')),
            'Precision': result.get('precision', result.get('mAP', {}).get('precision', 'N/A')),
            'Recall': result.get('recall', result.get('mAP', {}).get('recall', 'N/A')),
            'FPS': result.get('inference_speed', {}).get('fps', result.get('speed', {}).get('fps', 'N/A')),
            'Inference (ms)': result.get('inference_speed', {}).get('mean_time_ms', result.get('speed', {}).get('mean_ms', 'N/A')),
            'Total Detections': result.get('predictions', {}).get('total_detections', 'N/A'),
            'Avg Det/Image': result.get('predictions', {}).get('avg_detections_per_image', result.get('predictions', {}).get('avg_per_image', 'N/A'))
        }
        
        # Extract version and size
        version, size = extract_model_info(model_name)
        row['Version'] = version
        row['Size'] = size
        
        all_data.append(row)
    
    if not all_data:
        print("No valid data found after filtering!")
        return
    
    # Create DataFrame
    df = pd.DataFrame(all_data)
    
    # Convert numeric columns
    numeric_cols = ['mAP@50', 'mAP@50-95', 'mAP@75', 'Precision', 'Recall', 'FPS', 'Inference (ms)', 'Avg Det/Image']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Create markdown content
    markdown = "# YOLO WiderPerson Evaluation Results (PyTorch Models Only)\n\n"
    markdown += "*TensorRT models excluded from this comparison*\n\n"
    
    # 1. Global ranking by mAP@50
    markdown += "## 🏆 Global Ranking (All Models by mAP@50)\n\n"
    
    global_df = df.sort_values('mAP@50', ascending=False).copy()
    global_df['Rank'] = range(1, len(global_df) + 1)
    
    # Format for display
    display_cols = ['Rank', 'Model', 'mAP@50', 'mAP@50-95', 'mAP@75', 'Precision', 'Recall', 'FPS']
    global_display = global_df[display_cols].copy()
    
    # Format numeric columns
    for col in ['mAP@50', 'mAP@50-95', 'mAP@75', 'Precision', 'Recall']:
        global_display[col] = global_display[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else 'N/A')
    global_display['FPS'] = global_display['FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
    
    markdown += global_display.to_markdown(index=False)
    
    # 2. Grouped by YOLO version
    markdown += "\n\n## 📊 Results Grouped by YOLO Version\n\n"
    
    # Group by version
    versions = df['Version'].unique()
    versions = sorted(versions, key=lambda x: (int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 0))
    
    for version in versions:
        version_df = df[df['Version'] == version].copy()
        
        if len(version_df) == 0:
            continue
            
        markdown += f"\n### {version.upper()}\n\n"
        
        # Sort by mAP@50 within version
        version_df = version_df.sort_values('mAP@50', ascending=False)
        
        # Define size order
        size_order = {'n': 1, 's': 2, 'm': 3, 'l': 4, 'x': 5}
        version_df['size_order'] = version_df['Size'].map(size_order).fillna(6)
        version_df = version_df.sort_values(['mAP@50'], ascending=[False])
        
        # Prepare display
        version_display = version_df[['Model', 'Size', 'mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'FPS', 'Inference (ms)']].copy()
        
        # Format numeric columns
        for col in ['mAP@50', 'mAP@50-95', 'Precision', 'Recall']:
            version_display[col] = version_display[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else 'N/A')
        version_display['FPS'] = version_display['FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
        version_display['Inference (ms)'] = version_display['Inference (ms)'].apply(lambda x: f"{x:.2f}" if pd.notna(x) else 'N/A')
        
        # Map size to full name
        size_names = {'n': 'Nano', 's': 'Small', 'm': 'Medium', 'l': 'Large', 'x': 'Extra Large'}
        version_display['Size'] = version_display['Size'].map(size_names).fillna('Unknown')
        
        markdown += version_display.to_markdown(index=False)
    
    # 3. Performance Summary
    markdown += "\n\n## 📈 Performance Summary\n\n"
    
    # Best model per version
    markdown += "### Best Model per Version (by mAP@50)\n\n"
    
    best_per_version = []
    for version in versions:
        version_df = df[df['Version'] == version]
        if len(version_df) > 0:
            best = version_df.loc[version_df['mAP@50'].idxmax()]
            best_per_version.append({
                'Version': version.upper(),
                'Best Model': best['Model'],
                'mAP@50': f"{best['mAP@50']:.4f}" if pd.notna(best['mAP@50']) else 'N/A',
                'FPS': f"{best['FPS']:.1f}" if pd.notna(best['FPS']) else 'N/A'
            })
    
    best_df = pd.DataFrame(best_per_version)
    markdown += best_df.to_markdown(index=False)
    
    # 4. Size comparison across versions
    markdown += "\n\n### Model Size Comparison (Average mAP@50)\n\n"
    
    size_comparison = []
    for size in ['n', 's', 'm', 'l', 'x']:
        size_data = df[df['Size'] == size]
        if len(size_data) > 0:
            size_names = {'n': 'Nano', 's': 'Small', 'm': 'Medium', 'l': 'Large', 'x': 'Extra Large'}
            size_comparison.append({
                'Size': size_names.get(size, size),
                'Count': len(size_data),
                'Avg mAP@50': f"{size_data['mAP@50'].mean():.4f}",
                'Avg FPS': f"{size_data['FPS'].mean():.1f}",
                'Best mAP@50': f"{size_data['mAP@50'].max():.4f}"
            })
    
    if size_comparison:
        size_df = pd.DataFrame(size_comparison)
        markdown += size_df.to_markdown(index=False)
    
    # 5. Statistical Summary
    markdown += "\n\n## 📊 Statistical Summary\n\n"
    
    stats_data = {
        'Metric': ['mAP@50', 'mAP@50-95', 'FPS'],
        'Mean': [f"{df['mAP@50'].mean():.4f}", f"{df['mAP@50-95'].mean():.4f}", f"{df['FPS'].mean():.1f}"],
        'Std': [f"{df['mAP@50'].std():.4f}", f"{df['mAP@50-95'].std():.4f}", f"{df['FPS'].std():.1f}"],
        'Min': [f"{df['mAP@50'].min():.4f}", f"{df['mAP@50-95'].min():.4f}", f"{df['FPS'].min():.1f}"],
        'Max': [f"{df['mAP@50'].max():.4f}", f"{df['mAP@50-95'].max():.4f}", f"{df['FPS'].max():.1f}"]
    }
    
    stats_df = pd.DataFrame(stats_data)
    markdown += stats_df.to_markdown(index=False)
    
    # Save to file
    with open(output_file, 'w') as f:
        f.write(markdown)
    
    print(f"Results saved to {output_file}")
    
    # Also create CSV files
    # Global ranking CSV
    global_csv = output_file.replace('.md', '_global_ranking.csv')
    global_df.to_csv(global_csv, index=False)
    print(f"Global ranking CSV saved to {global_csv}")
    
    # Per version CSVs
    for version in versions:
        version_df = df[df['Version'] == version]
        if len(version_df) > 0:
            version_csv = output_file.replace('.md', f'_{version}.csv')
            version_df.to_csv(version_csv, index=False)
            print(f"{version} CSV saved to {version_csv}")

if __name__ == "__main__":
    create_markdown_tables("EvalResults", "widerperson_pytorch_results_new.md")

Results saved to widerperson_pytorch_results_new.md
Global ranking CSV saved to widerperson_pytorch_results_new_global_ranking.csv
yolov8 CSV saved to widerperson_pytorch_results_new_yolov8.csv
yolov9c CSV saved to widerperson_pytorch_results_new_yolov9c.csv
yolov9 CSV saved to widerperson_pytorch_results_new_yolov9.csv
yolov9t CSV saved to widerperson_pytorch_results_new_yolov9t.csv
yolov10 CSV saved to widerperson_pytorch_results_new_yolov10.csv
yolo11 CSV saved to widerperson_pytorch_results_new_yolo11.csv
yolo12 CSV saved to widerperson_pytorch_results_new_yolo12.csv


In [3]:
# compare_pytorch_tensorrt.py
import json
from pathlib import Path
import pandas as pd
import re

def extract_model_info(model_name):
    """Extract YOLO version and size from model name."""
    # Remove tensorrt suffix if present
    clean_name = model_name.lower().replace('_tensorrt', '').replace('-tensorrt', '').replace('tensorrt', '')
    
    # Pattern to match YOLOv8n, YOLOv11l, etc.
    pattern = r'(yolov?\d+)([nslmx])'
    match = re.search(pattern, clean_name)
    
    if match:
        version = match.group(1)
        size = match.group(2)
        return version, size, f"{version}{size}"
    return model_name, '', model_name

def create_comparison_tables(results_dir=".", output_file="pytorch_vs_tensorrt.md"):
    """
    Create comparison tables between PyTorch and TensorRT models.
    """
    
    # Find all JSON files
    pytorch_data = {}
    tensorrt_data = {}
    
    for folder in Path(results_dir).glob("*_results"):
        for json_file in folder.glob("*.json"):
            with open(json_file, 'r') as f:
                result = json.load(f)
            
            model_name = json_file.parent.name.replace("_widerperson_results", "")
            
            # Determine if it's TensorRT
            is_tensorrt = (
                'tensorrt' in model_name.lower() or
                result.get('is_tensorrt', False) or
                ('model' in result and 'tensorrt' in str(result['model']).lower()) or
                ('model' in result and '.engine' in str(result['model']))
            )
            
            # Extract base model info
            version, size, base_name = extract_model_info(model_name)
            
            data = {
                'model_name': model_name,
                'base_name': base_name,
                'version': version,
                'size': size,
                'mAP50': result.get('mAP50', result.get('mAP', {}).get('mAP50', None)),
                'mAP50-95': result.get('mAP50-95', result.get('mAP', {}).get('mAP50-95', None)),
                'precision': result.get('precision', result.get('mAP', {}).get('precision', None)),
                'recall': result.get('recall', result.get('mAP', {}).get('recall', None)),
                'fps': result.get('inference_speed', {}).get('fps', result.get('speed', {}).get('fps', None)),
                'inference_ms': result.get('inference_speed', {}).get('mean_time_ms', result.get('speed', {}).get('mean_ms', None)),
                'batch_size': result.get('inference_speed', {}).get('batch_size', result.get('speed', {}).get('batch_size', 1))
            }
            
            if is_tensorrt:
                tensorrt_data[base_name] = data
            else:
                pytorch_data[base_name] = data
    
    # Create comparison data
    comparisons = []
    
    for base_name in pytorch_data:
        if base_name in tensorrt_data:
            pt = pytorch_data[base_name]
            trt = tensorrt_data[base_name]
            
            comparison = {
                'Model': base_name.upper(),
                'Version': pt['version'].upper(),
                'Size': pt['size'].upper(),
                
                # PyTorch metrics
                'PT_mAP50': pt['mAP50'],
                'PT_FPS': pt['fps'],
                'PT_ms': pt['inference_ms'],
                
                # TensorRT metrics
                'TRT_FPS': trt['fps'],
                'TRT_ms': trt['inference_ms'],
                'TRT_Batch': trt['batch_size'],
                
                # Speedup calculations
                'FPS_Speedup': trt['fps'] / pt['fps'] if pt['fps'] and trt['fps'] else None,
                'ms_Speedup': pt['inference_ms'] / trt['inference_ms'] if pt['inference_ms'] and trt['inference_ms'] else None,
                
                # mAP difference (if available for TRT)
                'mAP_Diff': (trt['mAP50'] - pt['mAP50']) if trt['mAP50'] and pt['mAP50'] else None
            }
            
            comparisons.append(comparison)
    
    if not comparisons:
        print("No matching PyTorch-TensorRT pairs found!")
        return
    
    # Create DataFrame
    df = pd.DataFrame(comparisons)
    
    # Sort by version and size
    size_order = {'N': 1, 'S': 2, 'M': 3, 'L': 4, 'X': 5}
    df['size_order'] = df['Size'].map(size_order).fillna(6)
    df = df.sort_values(['Version', 'size_order'])
    df = df.drop('size_order', axis=1)
    
    # Create markdown content
    markdown = "# PyTorch vs TensorRT Performance Comparison\n\n"
    markdown += "*Comparing inference performance between PyTorch and TensorRT implementations*\n\n"
    
    # 1. Main comparison table
    markdown += "## 📊 Performance Comparison\n\n"
    
    # Format main comparison table
    main_df = df[['Model', 'PT_mAP50', 'PT_FPS', 'TRT_FPS', 'FPS_Speedup', 'PT_ms', 'TRT_ms', 'TRT_Batch']].copy()
    main_df.columns = ['Model', 'mAP@50', 'PyTorch FPS', 'TensorRT FPS', 'FPS Speedup', 'PT ms', 'TRT ms', 'TRT Batch']
    
    # Format numeric columns
    main_df['mAP@50'] = main_df['mAP@50'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else 'N/A')
    main_df['PyTorch FPS'] = main_df['PyTorch FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
    main_df['TensorRT FPS'] = main_df['TensorRT FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
    main_df['FPS Speedup'] = main_df['FPS Speedup'].apply(lambda x: f"{x:.2f}x" if pd.notna(x) else 'N/A')
    main_df['PT ms'] = main_df['PT ms'].apply(lambda x: f"{x:.2f}" if pd.notna(x) else 'N/A')
    main_df['TRT ms'] = main_df['TRT ms'].apply(lambda x: f"{x:.2f}" if pd.notna(x) else 'N/A')
    
    markdown += main_df.to_markdown(index=False)
    
    # 2. Speedup summary by version
    markdown += "\n\n## 🚀 Speedup Summary by Version\n\n"
    
    version_summary = []
    for version in df['Version'].unique():
        version_df = df[df['Version'] == version]
        
        avg_speedup = version_df['FPS_Speedup'].mean()
        min_speedup = version_df['FPS_Speedup'].min()
        max_speedup = version_df['FPS_Speedup'].max()
        
        version_summary.append({
            'Version': version,
            'Models': len(version_df),
            'Avg Speedup': f"{avg_speedup:.2f}x" if pd.notna(avg_speedup) else 'N/A',
            'Min Speedup': f"{min_speedup:.2f}x" if pd.notna(min_speedup) else 'N/A',
            'Max Speedup': f"{max_speedup:.2f}x" if pd.notna(max_speedup) else 'N/A'
        })
    
    version_summary_df = pd.DataFrame(version_summary)
    markdown += version_summary_df.to_markdown(index=False)
    
    # 3. Detailed comparison by model size
    markdown += "\n\n## 📏 Comparison by Model Size\n\n"
    
    size_names = {'N': 'Nano', 'S': 'Small', 'M': 'Medium', 'L': 'Large', 'X': 'Extra Large'}
    
    for size in ['N', 'S', 'M', 'L', 'X']:
        size_df = df[df['Size'] == size]
        if len(size_df) > 0:
            markdown += f"\n### {size_names.get(size, size)} Models\n\n"
            
            size_comparison = size_df[['Model', 'PT_FPS', 'TRT_FPS', 'FPS_Speedup', 'ms_Speedup']].copy()
            size_comparison.columns = ['Model', 'PyTorch FPS', 'TensorRT FPS', 'FPS Speedup', 'Latency Speedup']
            
            # Format
            size_comparison['PyTorch FPS'] = size_comparison['PyTorch FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
            size_comparison['TensorRT FPS'] = size_comparison['TensorRT FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
            size_comparison['FPS Speedup'] = size_comparison['FPS Speedup'].apply(lambda x: f"{x:.2f}x" if pd.notna(x) else 'N/A')
            size_comparison['Latency Speedup'] = size_comparison['Latency Speedup'].apply(lambda x: f"{x:.2f}x" if pd.notna(x) else 'N/A')
            
            markdown += size_comparison.to_markdown(index=False)
    
    # 4. Best speedups
    markdown += "\n\n## 🏆 Top Speedups\n\n"
    
    top_speedups = df.nlargest(10, 'FPS_Speedup')[['Model', 'PT_FPS', 'TRT_FPS', 'FPS_Speedup']].copy()
    top_speedups.columns = ['Model', 'PyTorch FPS', 'TensorRT FPS', 'Speedup']
    
    # Format
    top_speedups['PyTorch FPS'] = top_speedups['PyTorch FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
    top_speedups['TensorRT FPS'] = top_speedups['TensorRT FPS'].apply(lambda x: f"{x:.1f}" if pd.notna(x) else 'N/A')
    top_speedups['Speedup'] = top_speedups['Speedup'].apply(lambda x: f"{x:.2f}x" if pd.notna(x) else 'N/A')
    
    markdown += top_speedups.to_markdown(index=False)
    
    # 5. Summary statistics
    markdown += "\n\n## 📈 Summary Statistics\n\n"
    
    summary_stats = {
        'Metric': ['Average FPS Speedup', 'Average Latency Reduction', 'Models Compared'],
        'Value': [
            f"{df['FPS_Speedup'].mean():.2f}x",
            f"{(1 - 1/df['ms_Speedup'].mean()) * 100:.1f}%",
            len(df)
        ]
    }
    
    summary_df = pd.DataFrame(summary_stats)
    markdown += summary_df.to_markdown(index=False)
    
    # 6. Visualization data (for plotting)
    markdown += "\n\n## 📊 Visualization Data\n\n"
    markdown += "*Data formatted for easy plotting*\n\n"
    
    viz_df = df[['Model', 'PT_FPS', 'TRT_FPS', 'FPS_Speedup']].copy()
    viz_df = viz_df.sort_values('FPS_Speedup', ascending=False)
    
    # Format for plotting
    viz_df['PT_FPS'] = pd.to_numeric(viz_df['PT_FPS'], errors='coerce')
    viz_df['TRT_FPS'] = pd.to_numeric(viz_df['TRT_FPS'], errors='coerce')
    viz_df['FPS_Speedup'] = pd.to_numeric(viz_df['FPS_Speedup'], errors='coerce')
    
    markdown += "```python\n"
    markdown += "# Copy this data for plotting\n"
    markdown += f"models = {viz_df['Model'].tolist()}\n"
    markdown += f"pytorch_fps = {viz_df['PT_FPS'].fillna(0).tolist()}\n"
    markdown += f"tensorrt_fps = {viz_df['TRT_FPS'].fillna(0).tolist()}\n"
    markdown += f"speedups = {viz_df['FPS_Speedup'].fillna(0).tolist()}\n"
    markdown += "```\n"
    
    # Save to file
    with open(output_file, 'w') as f:
        f.write(markdown)
    
    print(f"Comparison saved to {output_file}")
    
    # Save CSV for further analysis
    csv_file = output_file.replace('.md', '.csv')
    df.to_csv(csv_file, index=False)
    print(f"CSV data saved to {csv_file}")

if __name__ == "__main__":
    create_comparison_tables("EvalResults", "pytorch_vs_tensorrt_comparison.md")

Comparison saved to pytorch_vs_tensorrt_comparison.md
CSV data saved to pytorch_vs_tensorrt_comparison.csv


In [4]:
# visualize_pytorch_tensorrt.py
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import re

def extract_model_info(model_name):
    """Extract YOLO version and size from model name."""
    clean_name = model_name.lower().replace('_tensorrt', '').replace('-tensorrt', '').replace('tensorrt', '')
    pattern = r'(yolov?\d+)([nslmx])'
    match = re.search(pattern, clean_name)
    
    if match:
        version = match.group(1)
        size = match.group(2)
        return version, size, f"{version}{size}"
    return model_name, '', model_name

def load_comparison_data(results_dir="."):
    """Load and process comparison data from JSON files."""
    pytorch_data = {}
    tensorrt_data = {}
    
    for folder in Path(results_dir).glob("*_results"):
        for json_file in folder.glob("*.json"):
            with open(json_file, 'r') as f:
                result = json.load(f)
            
            model_name = json_file.parent.name.replace("_widerperson_results", "")
            
            is_tensorrt = (
                'tensorrt' in model_name.lower() or
                result.get('is_tensorrt', False) or
                ('model' in result and '.engine' in str(result['model']))
            )
            
            version, size, base_name = extract_model_info(model_name)
            
            data = {
                'model_name': model_name,
                'base_name': base_name,
                'version': version,
                'size': size,
                'mAP50': result.get('mAP50', result.get('mAP', {}).get('mAP50', None)),
                'fps': result.get('inference_speed', {}).get('fps', result.get('speed', {}).get('fps', None)),
                'inference_ms': result.get('inference_speed', {}).get('mean_time_ms', result.get('speed', {}).get('mean_ms', None)),
            }
            
            if is_tensorrt:
                tensorrt_data[base_name] = data
            else:
                pytorch_data[base_name] = data
    
    # Create comparison dataframe
    comparisons = []
    for base_name in pytorch_data:
        if base_name in tensorrt_data:
            pt = pytorch_data[base_name]
            trt = tensorrt_data[base_name]
            
            comparisons.append({
                'Model': base_name.upper(),
                'Version': pt['version'],
                'Size': pt['size'].upper(),
                'PyTorch_FPS': pt['fps'],
                'TensorRT_FPS': trt['fps'],
                'PyTorch_ms': pt['inference_ms'],
                'TensorRT_ms': trt['inference_ms'],
                'Speedup': trt['fps'] / pt['fps'] if pt['fps'] and trt['fps'] else None,
                'mAP50': pt['mAP50']
            })
    
    return pd.DataFrame(comparisons)

def create_visualizations(df, output_dir="visualizations"):
    """Create all visualization plots."""
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # Set style
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    
    # 1. FPS Comparison Bar Chart
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(df))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df['PyTorch_FPS'], width, label='PyTorch', alpha=0.8)
    bars2 = ax.bar(x + width/2, df['TensorRT_FPS'], width, label='TensorRT', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('FPS (Frames Per Second)', fontsize=12)
    ax.set_title('PyTorch vs TensorRT FPS Comparison', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df['Model'], rotation=45, ha='right')
    ax.legend()
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.0f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'fps_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Speedup Factor Chart
    fig, ax = plt.subplots(figsize=(12, 8))
    
    df_sorted = df.sort_values('Speedup', ascending=True)
    colors = ['red' if x < 1 else 'green' for x in df_sorted['Speedup']]
    
    bars = ax.barh(df_sorted['Model'], df_sorted['Speedup'], color=colors, alpha=0.7)
    
    ax.axvline(x=1, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('Speedup Factor (TensorRT / PyTorch)', fontsize=12)
    ax.set_title('TensorRT Speedup Factors', fontsize=16, fontweight='bold')
    
    # Add value labels
    for bar, speedup in zip(bars, df_sorted['Speedup']):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{speedup:.2f}x', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'speedup_factors.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Latency Comparison
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(df))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df['PyTorch_ms'], width, label='PyTorch', alpha=0.8)
    bars2 = ax.bar(x + width/2, df['TensorRT_ms'], width, label='TensorRT', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Inference Time (ms)', fontsize=12)
    ax.set_title('Inference Latency Comparison', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df['Model'], rotation=45, ha='right')
    ax.legend()
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.1f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'latency_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 4. Speedup by Model Size
    fig, ax = plt.subplots(figsize=(10, 6))
    
    size_order = ['N', 'S', 'M', 'L', 'X']
    size_names = {'N': 'Nano', 'S': 'Small', 'M': 'Medium', 'L': 'Large', 'X': 'Extra Large'}
    
    speedup_by_size = df.groupby('Size')['Speedup'].agg(['mean', 'std']).reindex(size_order)
    
    x = range(len(speedup_by_size))
    ax.bar(x, speedup_by_size['mean'], yerr=speedup_by_size['std'], 
           capsize=5, alpha=0.7, color='skyblue', edgecolor='navy')
    
    ax.set_xlabel('Model Size', fontsize=12)
    ax.set_ylabel('Average Speedup Factor', fontsize=12)
    ax.set_title('Average TensorRT Speedup by Model Size', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([size_names.get(s, s) for s in speedup_by_size.index])
    
    # Add value labels
    for i, (idx, row) in enumerate(speedup_by_size.iterrows()):
        ax.text(i, row['mean'] + 0.05, f'{row["mean"]:.2f}x', 
                ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'speedup_by_size.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 5. Scatter Plot: FPS vs mAP
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot PyTorch models
    ax.scatter(df['PyTorch_FPS'], df['mAP50'], s=100, alpha=0.7, 
               label='PyTorch', marker='o', edgecolors='black')
    
    # Plot TensorRT models
    ax.scatter(df['TensorRT_FPS'], df['mAP50'], s=100, alpha=0.7, 
               label='TensorRT', marker='^', edgecolors='black')
    
    # Add model labels
    for idx, row in df.iterrows():
        ax.annotate(row['Model'], (row['PyTorch_FPS'], row['mAP50']), 
                   xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.7)
        ax.annotate(row['Model'], (row['TensorRT_FPS'], row['mAP50']), 
                   xytext=(5, -10), textcoords='offset points', fontsize=8, alpha=0.7)
    
    ax.set_xlabel('FPS (Frames Per Second)', fontsize=12)
    ax.set_ylabel('mAP@50', fontsize=12)
    ax.set_title('FPS vs Accuracy Trade-off', fontsize=16, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'fps_vs_map.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 6. Grouped Bar Chart by Version
    fig, ax = plt.subplots(figsize=(12, 8))
    
    versions = df['Version'].unique()
    version_data = []
    
    for version in versions:
        version_df = df[df['Version'] == version]
        version_data.append({
            'Version': version.upper(),
            'PyTorch_Avg': version_df['PyTorch_FPS'].mean(),
            'TensorRT_Avg': version_df['TensorRT_FPS'].mean(),
            'Avg_Speedup': version_df['Speedup'].mean()
        })
    
    version_df = pd.DataFrame(version_data)
    
    x = np.arange(len(version_df))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, version_df['PyTorch_Avg'], width, label='PyTorch Avg', alpha=0.8)
    bars2 = ax.bar(x + width/2, version_df['TensorRT_Avg'], width, label='TensorRT Avg', alpha=0.8)
    
    ax.set_xlabel('YOLO Version', fontsize=12)
    ax.set_ylabel('Average FPS', fontsize=12)
    ax.set_title('Average Performance by YOLO Version', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(version_df['Version'])
    ax.legend()
    
    # Add speedup text
    for i, row in version_df.iterrows():
        ax.text(i, max(row['PyTorch_Avg'], row['TensorRT_Avg']) + 5,
                f'{row["Avg_Speedup"]:.2f}x', ha='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'performance_by_version.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 7. Heatmap of Speedups
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create pivot table for heatmap
    heatmap_data = df.pivot_table(values='Speedup', index='Size', columns='Version', aggfunc='mean')
    
    # Reorder sizes
    size_order = ['N', 'S', 'M', 'L', 'X']
    heatmap_data = heatmap_data.reindex([s for s in size_order if s in heatmap_data.index])
    
    sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd', 
                cbar_kws={'label': 'Speedup Factor'}, ax=ax)
    
    ax.set_title('TensorRT Speedup Heatmap', fontsize=16, fontweight='bold')
    ax.set_xlabel('YOLO Version', fontsize=12)
    ax.set_ylabel('Model Size', fontsize=12)
    
    # Update y-axis labels
    size_names = {'N': 'Nano', 'S': 'Small', 'M': 'Medium', 'L': 'Large', 'X': 'Extra Large'}
    ax.set_yticklabels([size_names.get(s.get_text(), s.get_text()) for s in ax.get_yticklabels()])
    
    plt.tight_layout()
    plt.savefig(output_dir / 'speedup_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Visualizations saved to {output_dir}/")
    
    # Create summary statistics
    summary = f"""
    Summary Statistics:
    - Average PyTorch FPS: {df['PyTorch_FPS'].mean():.1f}
    - Average TensorRT FPS: {df['TensorRT_FPS'].mean():.1f}
    - Average Speedup: {df['Speedup'].mean():.2f}x
    - Max Speedup: {df['Speedup'].max():.2f}x ({df.loc[df['Speedup'].idxmax(), 'Model']})
    - Min Speedup: {df['Speedup'].min():.2f}x ({df.loc[df['Speedup'].idxmin(), 'Model']})
    """
    
    with open(output_dir / 'summary.txt', 'w') as f:
        f.write(summary)
    
    return summary

def create_advanced_visualizations(df, output_dir="visualizations"):
    """Create additional advanced visualizations."""
    
    # 8. Performance Improvement Distribution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Histogram of speedup factors
    ax1.hist(df['Speedup'], bins=15, alpha=0.7, color='green', edgecolor='black')
    ax1.axvline(df['Speedup'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Speedup"].mean():.2f}x')
    ax1.set_xlabel('Speedup Factor', fontsize=12)
    ax1.set_ylabel('Number of Models', fontsize=12)
    ax1.set_title('Distribution of TensorRT Speedups', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Box plot by model size
    size_order = ['N', 'S', 'M', 'L', 'X']
    df_ordered = df.copy()
    df_ordered['Size'] = pd.Categorical(df_ordered['Size'], categories=size_order, ordered=True)
    df_ordered = df_ordered.sort_values('Size')
    
    box_data = [df_ordered[df_ordered['Size'] == size]['Speedup'].values for size in size_order if size in df_ordered['Size'].values]
    box_labels = [size for size in size_order if size in df_ordered['Size'].values]
    
    bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    ax2.set_xlabel('Model Size', fontsize=12)
    ax2.set_ylabel('Speedup Factor', fontsize=12)
    ax2.set_title('Speedup Distribution by Model Size', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'speedup_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 9. Relative Performance Chart
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Calculate relative improvements
    df['Latency_Reduction'] = (1 - df['TensorRT_ms'] / df['PyTorch_ms']) * 100
    df_sorted = df.sort_values('Latency_Reduction', ascending=True)
    
    y_pos = np.arange(len(df_sorted))
    
    # Create horizontal bar chart
    bars = ax.barh(y_pos, df_sorted['Latency_Reduction'], alpha=0.7)
    
    # Color bars based on improvement
    for i, (bar, reduction) in enumerate(zip(bars, df_sorted['Latency_Reduction'])):
        if reduction > 50:
            bar.set_color('darkgreen')
        elif reduction > 30:
            bar.set_color('green')
        elif reduction > 10:
            bar.set_color('yellow')
        else:
            bar.set_color('orange')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_sorted['Model'])
    ax.set_xlabel('Latency Reduction (%)', fontsize=12)
    ax.set_title('TensorRT Latency Reduction Percentage', fontsize=16, fontweight='bold')
    
    # Add value labels
    for i, (idx, row) in enumerate(df_sorted.iterrows()):
        ax.text(row['Latency_Reduction'] + 1, i, f"{row['Latency_Reduction']:.1f}%", 
                va='center', fontsize=9)
    
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(output_dir / 'latency_reduction.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 10. Combined Performance Radar Chart
    from math import pi
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw=dict(projection='polar'))
    axes = axes.flatten()
    
    # Get unique versions
    versions = sorted(df['Version'].unique())
    
    for idx, version in enumerate(versions[:6]):  # Max 6 versions
        ax = axes[idx]
        version_df = df[df['Version'] == version]
        
        # Prepare data for radar chart
        categories = ['Nano', 'Small', 'Medium', 'Large', 'X-Large']
        size_map = {'N': 'Nano', 'S': 'Small', 'M': 'Medium', 'L': 'Large', 'X': 'X-Large'}
        
        pytorch_fps = []
        tensorrt_fps = []
        
        for cat in categories:
            size_key = [k for k, v in size_map.items() if v == cat][0]
            size_data = version_df[version_df['Size'] == size_key]
            
            if len(size_data) > 0:
                pytorch_fps.append(size_data['PyTorch_FPS'].values[0])
                tensorrt_fps.append(size_data['TensorRT_FPS'].values[0])
            else:
                pytorch_fps.append(0)
                tensorrt_fps.append(0)
        
        # Number of variables
        num_vars = len(categories)
        angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
        angles += angles[:1]
        
        pytorch_fps += pytorch_fps[:1]
        tensorrt_fps += tensorrt_fps[:1]
        
        # Plot
        ax.plot(angles, pytorch_fps, 'o-', linewidth=2, label='PyTorch', alpha=0.7)
        ax.fill(angles, pytorch_fps, alpha=0.25)
        
        ax.plot(angles, tensorrt_fps, 'o-', linewidth=2, label='TensorRT', alpha=0.7)
        ax.fill(angles, tensorrt_fps, alpha=0.25)
        
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories)
        ax.set_title(f'{version.upper()} Performance', fontsize=14, fontweight='bold', pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        ax.grid(True)
    
    # Hide unused subplots
    for idx in range(len(versions), 6):
        axes[idx].set_visible(False)
    
    plt.suptitle('Performance Comparison Across Model Sizes', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_dir / 'radar_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

def create_report_plots(results_dir=".", output_dir="visualizations"):
    """Main function to create all visualizations."""
    
    # Load data
    print("Loading comparison data...")
    df = load_comparison_data(results_dir)
    
    if df.empty:
        print("No comparison data found!")
        return
    
    print(f"Found {len(df)} model comparisons")
    
    # Create output directory
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # Create basic visualizations
    print("Creating basic visualizations...")
    summary = create_visualizations(df, output_dir)
    
    # Create advanced visualizations
    print("Creating advanced visualizations...")
    create_advanced_visualizations(df, output_dir)
    
    print(summary)
    
    # Create a simple HTML report
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>PyTorch vs TensorRT Comparison Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; }}
            h1 {{ color: #333; }}
            .plot {{ margin: 20px 0; text-align: center; }}
            img {{ max-width: 100%; height: auto; border: 1px solid #ddd; padding: 10px; }}
            .summary {{ background-color: #f0f0f0; padding: 20px; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <h1>PyTorch vs TensorRT Performance Comparison</h1>
        
        <div class="summary">
            <h2>Summary Statistics</h2>
            <pre>{summary}</pre>
        </div>
        
        <div class="plot">
            <h2>FPS Comparison</h2>
            <img src="fps_comparison.png" alt="FPS Comparison">
        </div>
        
        <div class="plot">
            <h2>Speedup Factors</h2>
            <img src="speedup_factors.png" alt="Speedup Factors">
        </div>
        
        <div class="plot">
            <h2>Latency Comparison</h2>
            <img src="latency_comparison.png" alt="Latency Comparison">
        </div>
        
        <div class="plot">
            <h2>Speedup by Model Size</h2>
            <img src="speedup_by_size.png" alt="Speedup by Size">
        </div>
        
        <div class="plot">
            <h2>FPS vs Accuracy Trade-off</h2>
            <img src="fps_vs_map.png" alt="FPS vs mAP">
        </div>
        
        <div class="plot">
            <h2>Performance by Version</h2>
            <img src="performance_by_version.png" alt="Performance by Version">
        </div>
        
        <div class="plot">
            <h2>Speedup Heatmap</h2>
            <img src="speedup_heatmap.png" alt="Speedup Heatmap">
        </div>
        
        <div class="plot">
            <h2>Speedup Distribution</h2>
            <img src="speedup_distribution.png" alt="Speedup Distribution">
        </div>
        
        <div class="plot">
            <h2>Latency Reduction</h2>
            <img src="latency_reduction.png" alt="Latency Reduction">
        </div>
        
        <div class="plot">
            <h2>Radar Comparison</h2>
            <img src="radar_comparison.png" alt="Radar Comparison">
        </div>
    </body>
    </html>
    """
    
    with open(output_dir / 'report.html', 'w') as f:
        f.write(html_content)
    
    print(f"\nHTML report saved to {output_dir}/report.html")
    print("All visualizations completed!")

if __name__ == "__main__":
    # Run the visualization pipeline
    create_report_plots(results_dir="EvalResults", output_dir="pytorch_tensorrt_visualizations")

Loading comparison data...
Found 21 model comparisons
Creating basic visualizations...
Visualizations saved to pytorch_tensorrt_visualizations/
Creating advanced visualizations...


/tmp/ipykernel_105003/3054210213.py:328: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)



    Summary Statistics:
    - Average PyTorch FPS: 48.1
    - Average TensorRT FPS: 111.8
    - Average Speedup: 2.63x
    - Max Speedup: 4.14x (YOLOV8X)
    - Min Speedup: 1.58x (YOLOV8N)
    

HTML report saved to pytorch_tensorrt_visualizations/report.html
All visualizations completed!
